# Chakra ViT fix: E. extra evaluation of finished runs (no training, ~10-30 min)

Re-scores every `runs/<tag>/best.pt` it finds: **third-source false alarms** (the 168 PICCOLO + PolypGen single frames that training could not read), **Stage-2 external sets** (PICCOLO, PolypGen C1-C6, PolypDB) and the 5 test sets at a **larger test resolution** (448 px; training stays at 352).

Settings: GPU T4 x2, Internet ON. Inputs: the `chakra_vit_fix` dataset (v1.1), `chakra-leakbench-v1`, `chakra-leakbench-e4-v1`, **`chakra-leakbench-s2-v1`**, and the **Output of each RUN_A / RUN_B / RUN_C notebook** (Add Input -> Your Work -> Notebooks). Run all; nothing to edit.

In [ ]:
# ---- 1. find this bundle and the data (works with /kaggle/input/<slug> and /kaggle/input/datasets/<owner>/<slug>)
import os, sys, glob, json, time, shutil, subprocess, zipfile
EXTRA_ROOTS = []      # lab PC only: e.g. [r"M:/chakramodelpro"]
SMOKE = os.environ.get("CHAKRA_SMOKE") == "1"   # developer switch: tiny random models (leave off)

hits = sorted(glob.glob("/kaggle/input/**/CHAKRA_BUNDLE_ID", recursive=True)) + \
       sorted(glob.glob(os.path.join(os.getcwd(), "**", "CHAKRA_BUNDLE_ID"), recursive=True))
assert hits, "bundle not found: add the uploaded chakra_vit_fix dataset as an Input"
BUNDLE = os.path.dirname(hits[0]); sys.path.insert(0, BUNDLE)
print("bundle:", BUNDLE, "|", open(hits[0]).read().strip())
import chakra_env
if EXTRA_ROOTS:
    roots, found = chakra_env.find_roots(EXTRA_ROOTS)
else:
    roots, found = chakra_env.find_roots()
for n in chakra_env.NEEDED:
    print(f"  {n}: {'found in ' + found[n] if n in found else 'MISSING'}")
assert roots and all(n in found for n in chakra_env.NEEDED), \
    "data not found: add BOTH chakra-leakbench-v1 and chakra-leakbench-e4-v1 as Inputs"
for m in ("v2.csv", "r6_mixneg.csv"):
    checked, missing = chakra_env.verify(os.path.join(BUNDLE, "manifests", m), roots)
    print(f"manifest {m}: {checked} rows sampled, {len(missing)} unresolved", missing[:2])
import torch
N_GPU = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPU)] or "NONE")
if N_GPU == 0 and not SMOKE:
    raise SystemExit("No GPU: Settings -> Accelerator -> GPU T4 x2")
W = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
LOGS = os.path.join(W, "logs"); os.makedirs(LOGS, exist_ok=True)

In [ ]:
# ---- 3b. re-score finished runs from their best.pt (no training) -------------------------------------
THIRD_SOURCE = True     # r6 runs: neg_thirdsource (PICCOLO + PolypGen single frames; needs chakra-leakbench-s2-v1)
EXTERNAL = True         # every split of s2_external.csv (PICCOLO, PolypGen C1-C6, PolypDB), if that dataset is attached
TEST_SIZES = [448]      # extra TEST resolutions for the 5 PraNet test sets ([] = skip); training size is unchanged

def dirs_upto(base, depth=4):
    base = os.path.abspath(base); start = base.rstrip(os.sep).count(os.sep)
    for dp, dns, _ in os.walk(base):
        if dp.count(os.sep) - start >= depth:
            dns[:] = []
        yield dp

def root_for(manifest, probe=20):
    # the folder under /kaggle/input from which this manifest's relative paths resolve
    import csv
    rels = [r["image"] for r in list(csv.DictReader(open(manifest)))[-probe:]]
    for d in dirs_upto("/kaggle/input" if os.path.isdir("/kaggle/input") else os.getcwd()):
        if any(os.path.exists(os.path.join(d, r)) for r in rels):
            return d
    return None

EVAL_ROOTS = list(roots)
r6_man = os.path.join(BUNDLE, "manifests", "r6_mixneg.csv")
import csv
third = [r for r in csv.DictReader(open(r6_man)) if r["split"] == "neg_thirdsource"]
tp = next((d for d in dirs_upto("/kaggle/input") if os.path.exists(os.path.join(d, third[0]["image"]))), None) if third else None
if tp and tp not in EVAL_ROOTS: EVAL_ROOTS.append(tp)
print("third-source frames:", "found under " + tp if tp else "NOT FOUND -> attach chakra-leakbench-s2-v1")
s2 = sorted(glob.glob("/kaggle/input/**/s2_external.csv", recursive=True))
s2_root = root_for(s2[0]) if s2 else None
if s2_root and s2_root not in EVAL_ROOTS: EVAL_ROOTS.append(s2_root)
print("s2_external.csv:", s2[0] if s2 else "not attached", "| resolves from", s2_root)

run_dirs = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/runs/*/best.pt", recursive=True)} |
                  {os.path.dirname(p) for p in glob.glob(os.path.join(W, "runs", "*", "best.pt"))})
print(len(run_dirs), "finished runs found:", [os.path.basename(d) for d in run_dirs])
assert run_dirs, "no runs/<tag>/best.pt found: Add Input -> the Output of the RUN_A / RUN_B / RUN_C notebooks"
EV = os.path.join(W, "eval_out"); jobs = []
for rd in run_dirs:
    tag = os.path.basename(rd); man = "r6_mixneg.csv" if tag.startswith("r6_mixneg") else "v2.csv"
    base = [sys.executable, os.path.join(BUNDLE, "eval_only.py"), "--run-dir", rd, "--data-root", *EVAL_ROOTS, "--out", EV]
    if THIRD_SOURCE and man == "r6_mixneg.csv" and tp:
        jobs.append((f"{tag}__third", base + ["--manifest", os.path.join(BUNDLE, "manifests", man), "--splits", "neg_thirdsource"]))
    for sz in TEST_SIZES:
        tests = ["test_kvasir", "test_clinicdb", "test_colondb", "test_etis", "test_cvc300"]
        jobs.append((f"{tag}__size{sz}", base + ["--manifest", os.path.join(BUNDLE, "manifests", man), "--splits", *tests, "--size", str(sz)]))
    if EXTERNAL and s2 and s2_root:
        jobs.append((f"{tag}__s2ext", base + ["--manifest", s2[0], "--out", os.path.join(W, "eval_out_s2")]))
if SMOKE:
    jobs = [(n, c + ["--workers", "0"]) for n, c in jobs]
print(len(jobs), "eval jobs")

def run_queue(jobs, n_slots):
    free, live = list(range(max(1, n_slots))), []
    while jobs or live:
        while jobs and free:
            name, cmd = jobs.pop(0); g = free.pop(0)
            env = dict(os.environ, **({"CUDA_VISIBLE_DEVICES": str(g)} if n_slots > 1 else {}))
            log = open(os.path.join(LOGS, name + ".log"), "w")
            live.append((name, subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env), log, g))
            print("start", name, flush=True)
        time.sleep(1 if SMOKE else 10)
        for item in list(live):
            name, p, log, g = item
            if p.poll() is not None:
                live.remove(item); free.append(g); log.close()
                print(f"done {name}: exit {p.returncode}", open(os.path.join(LOGS, name + ".log"), errors="ignore").read()[-300:] if p.returncode else "", flush=True)

run_queue(jobs, N_GPU)

In [ ]:
# ---- 4. tables + SEND_BACK_EVAL.zip ---------------------------------------------------------------------
res = []
for f in sorted(glob.glob(os.path.join(W, "eval_out*", "*", "eval_results.json"))):
    r = json.load(open(f))
    for k, t in r["tests"].items():
        res.append(dict(run=r["tag"], test_size=r["test_size"], split=k, n=t.get("n"),
                        mDice=t.get("mDice"), small=t.get("mDice_small"), FA=t.get("false_alarm_rate@0.001")))
import pandas as pd
T = pd.DataFrame(res)
if len(T):
    pd.set_option("display.width", 200)
    print(T.sort_values(["run", "test_size", "split"]).to_string(index=False))
zp = os.path.join(W, "SEND_BACK_EVAL.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(os.path.join(W, "eval_out*", "*", "*.json")) + glob.glob(os.path.join(LOGS, "*.log")):
        z.write(f, os.path.relpath(f, W))
print("->", zp, f"({os.path.getsize(zp) / 1e6:.1f} MB)")

**Download** `SEND_BACK_EVAL.zip` from the Output tab and send it back.